# CV classification: frozen-snapshot evaluation

Labels follow the recorded Gaia Science Alerts classes. CV is the positive class.
This is an inspired pipeline, not a reproduction of the published scores.

## Experiment design

Retained 7,260 alerts in 7,257 groups; related alerts stay in the same split.
Seed: 42. Outer stratified group folds assign 60% to training, 20% to validation,
and 20% to testing.

| Split | Samples | CVs | Non-CVs |
| --- | --- | --- | --- |
| Training | 4356 | 477 | 3879 |
| Validation | 1452 | 159 | 1293 |
| Test | 1452 | 159 | 1293 |

Four formal experiments compare Random Forest and LightGBM using either nine
light-curve features or thirteen features. The four additional Gaia DR3 features
are `bp_rp`, `parallax`, `parallax_over_error`, and `pm`.

| Model | Algorithm | Features |
| --- | --- | --- |
| rf_9 | Random Forest | 9 light-curve features |
| lgbm_9 | LightGBM | 9 light-curve features |
| rf_13 | Random Forest | 9 light-curve + 4 Gaia DR3 features |
| lgbm_13 | LightGBM | 9 light-curve + 4 Gaia DR3 features |

Each formal experiment evaluates eight settings with three training-only group
folds using mean AP. Median imputation is fitted inside each training fold.
Validation selects an F1 threshold, breaking ties by precision and then higher
threshold. The recommended model is selected by validation AP, then validation
F1. No refit on training plus validation is performed after threshold selection.

Training fits the models and selects hyperparameters through internal
cross-validation. Validation selects thresholds and the recommended model;
testing evaluates the locked choice.

## Validation performance

All four models are evaluated on the same 1,452 validation objects, including
159 CVs. Precision, recall and F1 use each model's validation-selected threshold.
AP and ROC-AUC measure ranking across thresholds.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

REPORTS = Path.cwd() if Path.cwd().name == 'reports' else Path.cwd() / 'reports'
metrics = pd.read_csv(REPORTS / 'metrics.csv')
columns = ['model', 'precision', 'recall', 'f1', 'ap', 'roc_auc', 'threshold']
validation_metrics = metrics.loc[metrics['split'].eq('validation') & metrics['kind'].eq('formal'), columns].copy()
validation_metrics.columns = ['Model', 'Precision', 'Recall', 'F1', 'AP', 'ROC-AUC', 'Threshold']
display(validation_metrics.style.hide(axis='index').format(precision=4))

Model,Precision,Recall,F1,AP,ROC-AUC,Threshold
rf_9,0.7950,0.8050,0.8000,0.8836,0.9763,0.2940
lgbm_9,0.8797,0.7358,0.8014,0.8762,0.9717,0.4045
rf_13,0.9318,0.7736,0.8454,0.9199,0.9846,0.6991
lgbm_13,0.9236,0.8365,0.8779,0.9443,0.9894,0.4112


We report average precision (AP). Validation F1 is used to select thresholds.
The validation results support model selection.

The final selected model for test evaluation is **lgbm_13**, with validation AP
**0.9443**, F1 **0.8779**, and a locked threshold of **0.411169**.

The confusion matrices use each model's validation-selected threshold. The PR
and ROC curves use validation probabilities across thresholds.

![Validation confusion matrices](validation_confusion_matrices.png)

![Validation precision-recall and ROC curves](validation_pr_roc_curves.png)

## Permutation importance

Permutation importance is measured on validation data. Each feature is shuffled
ten times with seed 42 while the fitted model stays fixed. Importance is the
mean decrease in AP; larger decreases indicate greater model reliance.

![Validation permutation importance](permutation_importance.png)

The three largest mean validation AP decreases for each model are:

| Model | Most important | Second | Third |
| --- | --- | --- | --- |
| rf_9 | time_span_days (0.2249) | n_peaks_rm_2_7 (0.1172) | bright_excursion_mag (0.1066) |
| lgbm_9 | n_peaks_rm_2_7 (0.3424) | time_span_days (0.2424) | bright_excursion_mag (0.1165) |
| rf_13 | pm (0.1376) | time_span_days (0.0871) | bright_excursion_mag (0.0364) |
| lgbm_13 | time_span_days (0.1900) | pm (0.1326) | n_peaks_rm_2_7 (0.0668) |

Permutation importance is not causal. Correlated features may share importance.
Time span can reflect observation history and sample selection as well as source
behaviour. These results are not used to revise features after testing.

## Test performance of the selected model

Validation metrics selected **lgbm_13**: LightGBM with nine light-curve and four
Gaia DR3 features. Its threshold was fixed at **0.411169** using validation F1.
The saved model and threshold are applied to 1,452 test objects, including
159 CVs, without refitting or further threshold selection.

In [2]:
test_table = metrics.loc[metrics['split'].eq('test'), columns].copy()
test_table.columns = ['Model', 'Precision', 'Recall', 'F1', 'AP', 'ROC-AUC', 'Threshold']
display(test_table.style.hide(axis='index').format(precision=4))

Model,Precision,Recall,F1,AP,ROC-AUC,Threshold
lgbm_13,0.8889,0.8553,0.8718,0.9392,0.9863,0.4112


The selected model identifies **136 of 159 CVs**, with **17 false positives**
and **23 false negatives**. Test precision is **88.89%**, recall **85.53%**,
F1 **0.8718**, AP **0.9392**, and ROC-AUC **0.9863**.

The confusion matrix uses the locked threshold. The PR and ROC curves use the
same test probabilities across thresholds.

![Selected model test confusion matrix](test_confusion_matrix.png)

![Selected model test precision-recall and ROC curves](test_pr_roc_curves.png)

## Error light curves from the test set

The gallery shows the ten highest-scoring false positives and ten lowest-scoring
false negatives of the selected **lgbm_13** model on the test set. These are
selected diagnostic examples, not a random sample of all errors.

![Selected model test error light curves](error_lightcurves.png)